<a href="https://colab.research.google.com/github/VamikaSinghal/mh-skin-enhancer/blob/main/%5Blatest%5D_comfyui_klein9B_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ComfyUI — klein-9B Skin Enhancer

**Before running:**
1. Runtime → Change runtime type → **GPU (A100/L4)**.
2. Add your **HF_TOKEN** in the 🔑 Secrets tab (and toggle notebook access ON).
3. Accept the klein-9B license once at https://huggingface.co/black-forest-labs/FLUX.2-klein-9B → 'Agree and access'.

Then run cells **1 → 4** top to bottom. Everything installs into `/content/ComfyUI` (local disk).
On a fresh runtime, just re-run all four — no manual file copying.

_Cell 4 stays running and embeds the ComfyUI UI inline — scroll down inside its output to use it._

In [ ]:
# CELL 1 — Install ComfyUI + Manager + dependencies
import os
COMFY = '/content/ComfyUI'
if not os.path.isdir(COMFY):
    !git clone https://github.com/comfyanonymous/ComfyUI {COMFY}
%cd {COMFY}
!pip install -q -r requirements.txt           # installs comfy_aimdo etc. (fixes the old ModuleNotFound)
if not os.path.isdir(f'{COMFY}/custom_nodes/comfyui-manager'):
    !git clone https://github.com/Comfy-Org/ComfyUI-Manager {COMFY}/custom_nodes/comfyui-manager
print('✅ ComfyUI + Manager ready')

Cloning into '/content/ComfyUI'...
remote: Enumerating objects: 41656, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 41656 (delta 8), reused 6 (delta 6), pack-reused 41646 (from 2)
Receiving objects: 100% (41656/41656), 83.01 MiB | 16.40 MiB/s, done.
Resolving deltas: 100% (28253/28253), done.
/content/ComfyUI
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.6/26.6 MB 103.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 143.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 127.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.8/346.8 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.8/63.8 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.6/87.6 MB 10.4 MB/s

In [ ]:
# CELL 2 — Custom nodes the workflow needs
import os, subprocess
CN = '/content/ComfyUI/custom_nodes'
repos = {
    'ComfyUI_essentials':            'https://github.com/cubiq/ComfyUI_essentials',
    'comfyui_face_parsing':          'https://github.com/Ryuukeisyou/comfyui_face_parsing',
    'ComfyUI_LayerStyle':            'https://github.com/chflame163/ComfyUI_LayerStyle',
    'ComfyUI-Impact-Pack':           'https://github.com/ltdrdata/ComfyUI-Impact-Pack',
    'ComfyUI-KJNodes':               'https://github.com/kijai/ComfyUI-KJNodes',
    'ComfyUI-SeedVR2_VideoUpscaler': 'https://github.com/numz/ComfyUI-SeedVR2_VideoUpscaler',
}
for folder, url in repos.items():
    p = os.path.join(CN, folder)
    if os.path.isdir(p): print('⏭️', folder)
    else: print('⬇️', folder); subprocess.run(['git','clone','--depth','1',url,p], check=False)
for folder in repos:
    req = os.path.join(CN, folder, 'requirements.txt')
    if os.path.exists(req): subprocess.run(['pip','install','-q','-r',req], check=False)
print('✅ custom nodes ready (face_parsing + seedvr2 download their model weights on first run)')

⬇️ ComfyUI_essentials
⬇️ comfyui_face_parsing
⬇️ ComfyUI_LayerStyle
⬇️ ComfyUI-Impact-Pack
⬇️ ComfyUI-KJNodes
⬇️ ComfyUI-SeedVR2_VideoUpscaler
✅ custom nodes ready (face_parsing + seedvr2 download their model weights on first run)


In [ ]:
# CELL 3 — Models: klein-9B + qwen_3_8b + VAE + the two klein LoRAs
import os, shutil
os.environ["HF_HUB_DISABLE_XET"] = "1"


from huggingface_hub import hf_hub_download, list_repo_files
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
C = '/content/ComfyUI'
for d in ['diffusion_models', 'text_encoders', 'vae', 'loras']:
    os.makedirs(f'{C}/models/{d}', exist_ok=True)

def pull(repo, fname, dest, token=None):
    p = hf_hub_download(repo, fname, local_dir=dest, token=token)
    flat = os.path.join(dest, os.path.basename(fname))
    if os.path.abspath(p) != os.path.abspath(flat):
        shutil.move(p, flat)
    if '/' in fname:
        sub = os.path.join(dest, fname.split('/')[0])
        if os.path.isdir(sub): shutil.rmtree(sub, ignore_errors=True)
    return flat

# klein-9B diffusion model (GATED — needs HF_TOKEN + accepted license)
pull('black-forest-labs/FLUX.2-klein-9B', 'flux-2-klein-9b.safetensors', f'{C}/models/diffusion_models', HF_TOKEN)
# qwen_3_8b text encoder
pull('Comfy-Org/flux2-klein-9B', 'split_files/text_encoders/qwen_3_8b_fp8mixed.safetensors', f'{C}/models/text_encoders', HF_TOKEN)
# VAE
pull('Comfy-Org/flux2-dev', 'split_files/vae/flux2-vae.safetensors', f'{C}/models/vae')

# --- the two klein LoRAs, renamed to the names the v5 workflow expects ---
def grab_lora(repo, out_name, prefer=None):
    files = [f for f in list_repo_files(repo)
             if f.endswith('.safetensors') and 'checkpoint' not in f]
    pick = next((f for f in files if prefer and prefer in f), None) or files[-1]
    src = hf_hub_download(repo, pick, local_dir=f'{C}/models/loras')
    dst = f'{C}/models/loras/{out_name}'
    if os.path.abspath(src) != os.path.abspath(dst):
        shutil.move(src, dst)
    print(f'✅ LoRA: {out_name}  (from {pick})')

grab_lora('dx8152/Flux2-Klein-9B-Enhanced-Details', 'klein-enhanced-details.safetensors')      # realistic.safetensors
grab_lora('linoyts/Flux2-Klein-Delight-LoRA',       'klein-delight.safetensors', prefer='v2')   # pytorch_lora_weights_v2.safetensors

!ls -lh {C}/models/diffusion_models {C}/models/text_encoders {C}/models/vae {C}/models/loras

flux-2-klein-9b.safetensors:   0%|          | 0.00/18.2G [00:00<?, ?B/s]

qwen_3_8b_fp8mixed.safetensors:   0%|          | 0.00/8.66G [00:00<?, ?B/s]

flux2-vae.safetensors:   0%|          | 0.00/336M [00:00<?, ?B/s]

realistic.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

✅ LoRA: klein-enhanced-details.safetensors  (from realistic.safetensors)


pytorch_lora_weights_v2.safetensors:   0%|          | 0.00/127M [00:00<?, ?B/s]

✅ LoRA: klein-delight.safetensors  (from pytorch_lora_weights_v2.safetensors)
/content/ComfyUI/models/diffusion_models:
total 17G
-rw-r--r-- 1 root root 17G Jun 24 19:38 flux-2-klein-9b.safetensors
-rw-r--r-- 1 root root   0 Jun 24 19:22 put_diffusion_model_files_here

/content/ComfyUI/models/loras:
total 437M
-rw-r--r-- 1 root root 121M Jun 24 19:39 klein-delight.safetensors
-rw-r--r-- 1 root root 317M Jun 24 19:39 klein-enhanced-details.safetensors
-rw-r--r-- 1 root root    0 Jun 24 19:22 put_loras_here

/content/ComfyUI/models/text_encoders:
total 8.1G
-rw-r--r-- 1 root root    0 Jun 24 19:22 put_text_encoder_files_here
-rw-r--r-- 1 root root 8.1G Jun 24 19:39 qwen_3_8b_fp8mixed.safetensors

/content/ComfyUI/models/vae:
total 321M
-rw-r--r-- 1 root root 321M Jun 24 19:39 flux2-vae.safetensors
-rw-r--r-- 1 root root    0 Jun 24 19:22 put_vae_here


In [ ]:
# CELL 4 — Launch ComfyUI (embedded inline). KEEP RUNNING. Scroll down in this cell's output for the UI.
import threading, time, socket
%cd /content/ComfyUI
def serve(port):
    while True:
        time.sleep(0.5)
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        if s.connect_ex(('127.0.0.1', port)) == 0: s.close(); break
        s.close()
    from google.colab import output
    output.serve_kernel_port_as_iframe(port, height=1024)
threading.Thread(target=serve, daemon=True, args=(8188,)).start()
!python main.py --dont-print-server --enable-cors-header

/content/ComfyUI
[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[START] Security scan
[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-06-24 19:39:16.239
** Platform: Linux
** Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
** Python executable: /usr/bin/python3
** ComfyUI Path: /content/ComfyUI
** ComfyUI Base Folder Path: /content/ComfyUI
** User directory: /content/ComfyUI/user
** ComfyUI-Manager config path: /content/ComfyUI/user/__manager/config.ini
** Log path: /content/ComfyUI/user/comfyui.log
[INFO] 
Prestartup times for custom nodes:
[INFO]    6.2 seconds: /content/ComfyUI/custom_nodes/comfyui-manager
[INFO] 
[WARNING] WARNING: You need pytorch with cu130 or hig

<IPython.core.display.Javascript object>

FETCH ComfyRegistry Data: 5/156
FETCH ComfyRegistry Data: 10/156
FETCH ComfyRegistry Data: 15/156
FETCH ComfyRegistry Data: 20/156
FETCH ComfyRegistry Data: 25/156
FETCH ComfyRegistry Data: 30/156
FETCH ComfyRegistry Data: 35/156
FETCH ComfyRegistry Data: 40/156
FETCH ComfyRegistry Data: 45/156
FETCH ComfyRegistry Data: 50/156
FETCH ComfyRegistry Data: 55/156
FETCH ComfyRegistry Data: 60/156
FETCH ComfyRegistry Data: 65/156
FETCH ComfyRegistry Data: 70/156
FETCH ComfyRegistry Data: 75/156
FETCH ComfyRegistry Data: 80/156
[WARNING] [DEPRECATION WARNING] Detected import of deprecated legacy API: /scripts/ui.js. This is likely caused by a custom node extension using outdated APIs. Please update your extensions or contact the extension author for an updated version.
[WARNING] [DEPRECATION WARNING] Detected import of deprecated legacy API: /extensions/core/clipspace.js. This is likely caused by a custom node extension using outdated APIs. Please update your extensions or contact the extensi

In [ ]:
!df -h /
!du -sh /content/ComfyUI/models/* 2>/dev/null | sort -h